# Simulation of Argon in Gaseous and Liquid Phases
> Harrison B. Prosper<br>
> Feb. 21 2026

## Introduction
In this notebook, which is inspired by the beautiful results by Rahman [1], we simulate a microscopic spherical container of argon atoms.
We assume that the potential energy function of two argon atoms can be modeled using the **Lennard-Jones** (LJ) potential [2,3]), 
\begin{equation}
    V(r) = 4 \epsilon \left[ \left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^6 \right|,\tag{1}
\end{equation}
a widely used model of the interaction between two neutral atoms. The quantity $r = ||\mathbf{r}_i - \mathbf{r}_j||$  is the distance between the atoms, where $\mathbf{r}_i$ and $\mathbf{r}_j$ are the position vectors of atoms $i$ and $j$, respectively; 
$\epsilon$ is an energy that characterizes the interaction and $\sigma$ is the distance at which (by convention) $V(r) = 0$. By the same convention, $V(r)$ is also zero at $r = \infty$.
The first term in the potential yields a **repulsive force**, while the second term yields an **attractive force**. Because the potential falls off rapidly with the distance $r$ between atoms the latter are essentially free particles when their separation is much greater than $\sigma$. 

**Note**: The mathematical details below are not necessary to understand this notebook! They are presented here for completeness. To declutter the notation, we'll use the bold script notation for vectors.

## Force between argon atoms

The force on atom $i$ due to atom $j$ is given by $\mathbf{F}_i(r) = - \nabla_{r_i} V(r)$, where $\nabla_{r_i} V$ is a called the **gradient** of $V(r)$. Using the result
\begin{align}
    - \nabla_{r_i} \frac{1}{r^n} & = n \frac{\hat{r}}{r^{n+1}},\tag{2}
\end{align}
the Lennard-Jones (LJ) force on atom $i$ due to atom $j$ is given by,
\begin{align}
    \mathbf{F}_i(r) & = 24 \left(\frac{\epsilon}{\sigma}\right)\left[ 2 \left(\frac{\sigma}{r}\right)^{13} - \left(\frac{\sigma}{r}\right)^7\right] \hat{r}, \text{ where}\nonumber\\
    \hat{r} & = \frac{\mathbf{r}_i - \mathbf{r}_j}{r},\tag{3}
\end{align}
is a unit vector that points from atom $j$ to atom $i$.

In simulations it is usually a good idea to write all quantities in terms of small dimensionless numbers (that is, numbers without units). Consider Newton's 2nd law applied to the LJ force, 
\begin{align}
    \mathbf{F}  &= m \frac{d\mathbf{v}}{dt},\nonumber\\
    24 \left(\frac{\epsilon}{\sigma}\right)\left[ 2 \left(\frac{\sigma}{r}\right)^{13} - \left(\frac{\sigma}{r}\right)^7\right]\hat{r} & = m \frac{d\mathbf{v}}{dt}. 
\end{align}
The above can be rewritten as 
\begin{align}
    24 \left[ 2 \left(\frac{1}{r/\sigma}\right)^{13} - \left(\frac{1}{r/\sigma}\right)^7\right]\hat{r} & =   \left(\frac{\sigma m}{\epsilon}\right) \frac{d\mathbf{v}}{dt}. \tag{4}
\end{align}
Notice that in Eq.(4), $v_0 = \sqrt{\epsilon / m}$ has units of velocity. Its value is $159 \text{ m/s}$ in SI units, while $t_0 = \sigma / v_0$ has units of time and has the value $2.14 \times 10^{-12} \text{ s}$. Therefore, we can write
\begin{align}
    24 \left[ 2 \left(\frac{1}{r/\sigma}\right)^{13} - \left(\frac{1}{r/\sigma}\right)^7\right]\hat{r} & =   \left(\frac{\sigma}{\epsilon / m}\right) \frac{d\mathbf{v}}{dt}, \nonumber\\
    & =   \left(\frac{\sigma}{v_0^2}\right) \frac{d\mathbf{v}}{dt}, \nonumber\\
    & =   \left(\frac{t_0}{v_0}\right) \frac{d\mathbf{v}}{dt}, \nonumber\\
   24 \left[ 2 \left(\frac{1}{r/\sigma}\right)^{13} - \left(\frac{1}{r/\sigma}\right)^7\right]\hat{r}  & =    \frac{d\mathbf{v}/v_0}{dt/t_0} .\tag{5}
\end{align}
Defining the dimensionless (that is, "unit-less") variables $r^* = r / \sigma$, $t^* = t / t_0$, and $v^* = v / v_0$, we can write Newton's 2nd law for this problem as
\begin{align}
    \mathbf{f}^* & = \frac{d\mathbf{v}^*}{dt^*} , \text{ where}\\[6pt]
    \mathbf{f}^* & = f(r^*) \, \hat{r},\text{ and}\nonumber\\[6pt]
   f(r^*) & = 24 \left[ 2 \left(\frac{1}{r^*}\right)^{13} - \left(\frac{1}{r^*}\right)^7\right].\tag{6}
\end{align}
For simplicity, hereafter, we'll drop the superscript "*" on the variables with the understanding that $r$, $t$, and $v$ are dimensionless.

## Predictor-corrector formulae
Given the current positions and velocities of all particles and the net force on every particle their future positions and velocities can  be predicted using  Newton's 2nd law. However, for all but the simplest systems, the equations can only be solved approximately. The starting point are the Taylor series expansions
\begin{align}
    \mathbf{r}(t + h) & = \mathbf{r}(t) + \mathbf{v}(t) \, h  + \frac{1}{2} \mathbf{f}(t)  \, h^2  + {\cal O}(h^3),\tag{7}\\
    \mathbf{v}(t + h) & = \mathbf{v}(t) + \mathbf{f}(t)  \, h   + \frac{1}{2} \dot{\mathbf{f}}(t)  \, h^2 + {\cal O}(h^3) \tag{8},
\end{align}
where $h$ is a small time interval and $\mathbf{f}(t)$ is the **force per unit mass**. The symbol ${\cal O}(h^3)$ means "terms of order $h^3$", that is, terms that are proportional to $h^3$ or higher powers of $h$. The approximation arises when we neglect these "higher order terms", presumably because we judge them to be small enough to be neglected. 

Equations (7) and (8) are accurate to ${\cal O}(h^3)$. Therefore, if $h = 10^{-3}$ in dimensionless time units, the largest neglected term would  be proportional to $10^{-9}$. But to proceed we need an expression for the rate of change of the force per unit mass,  $\mathbf{j} \equiv \dot{\mathbf{f}}(t)$, which is sometimes referred to as the "jerk"! 

The standard approach for improving the predictions is first to predict the positions and velocities using Eqs. (7) and (8) by neglecting the jerk term in Eq.(8) and then correcting the predictions to achieve the desired accuracy. Methods that use this approach are called **predictor-corrector** methods. We shall derive a method that produces predictions for both the positions and the velocities that are accurate to ${\cal O}(h^3)$. 

### 3rd-order predictor-corrector
Let's simplify the notation 
\begin{align}
    \mathbf r_n & = \mathbf{r}(t) , &\quad 
    \mathbf r_{n+1}  = \mathbf{r}(t + h) ,\nonumber\\
    \mathbf v_n & = \mathbf{v}(t) , &\quad 
    \mathbf v_{n+1}  = \mathbf{v}(t + h) ,\nonumber\\
    \mathbf f_n & = \mathbf{f}(t) , &\quad 
    \mathbf f_{n+1}  = \mathbf{f}(t + h), \tag{9}
\end{align}
and write Eqs.(7) and (8) as follows,
\begin{align}
    \mathbf{r}_{n+1} & = \mathbf{r}_n + \mathbf{v}_n h + \frac{1}{2}\mathbf{f}_n h^2 + {\cal O}(h^3),\tag{15}\\
    \mathbf{v}_{n+1} & = \mathbf{v}_n + \mathbf{f}_n h + \frac{1}{2} \mathbf{j}_n h^2  + {\cal O}(h^3),\tag{16}\\
\end{align}
First, let's Taylor expand $\mathbf{j}_{n+1} \equiv \mathbf{j}(t+h)$ up to ${\cal O}(h^2)$,
\begin{align}
    \mathbf{f}_{n+1} & = \mathbf{f}_n + \mathbf{j}_n h + {\cal O}(h^2),\tag{17}
\end{align}
and rewrite it as
\begin{align}
    \mathbf{j}_n h & = \mathbf{f}_{n+1} - \mathbf{f}_n + {\cal O}(h^2).\tag{18}
\end{align}
If Eq.(18) is substituted into the expression for $\mathbf{v}_{n+1}$, Eq.(16), this yields
\begin{align}
\mathbf{v}_{n+1} & = \mathbf{v}_n + \mathbf{f}_n h + \frac{1}{2}(\mathbf{f}_{n+1} - \mathbf{f}_n + {\cal O}(h^2)) h,\nonumber\\
        &= \mathbf{v}_n + \frac{h}{2}(\mathbf{f}_n + \mathbf{f}_{n+1}) + {\cal O}(h^3),\tag{19}
\end{align}
an expression for the velocity at time $t = t + h$ that achieves the desired accuracy namely ${\cal O}(h^3)$. If Eq.(18) is substituted into a more accurate expression for $\mathbf{r}_{n+1}$ we find 
\begin{align}
    \mathbf{r}_{n+1} & = \mathbf{r}_n + \mathbf{v}_n h + \frac{1}{2}\mathbf{f}_n h^2 +  \frac{1}{6}\mathbf{j}_n h^3 + {\cal O}(h^4), \nonumber\\
     & = \mathbf{r}_n + \mathbf{v}_n h + \frac{1}{2}\mathbf{f}_n h^2 +  \frac{1}{6}(\mathbf{f}_{n+1} - \mathbf{f}_n + {\cal O}(h^2))h^2 + {\cal O}(h^4), \nonumber\\
    & = \mathbf{r}_n + h \mathbf{v}_n  +  \frac{h^2}{6}(\mathbf{f}_{n+1} + 2 \mathbf{f}_n) + {\cal O}(h^4). \nonumber
\end{align}
Inspired by the elegant form of Eq.(19) (and ChatGPT 5.2!),
consider the difference $\mathbf{r}_{n+1} - \frac{h}{2}(\mathbf{v}_n + \mathbf{v}_{n+1})$
\begin{align}
    \mathbf{r}_{n+1} - \frac{h}{2}(\mathbf{v}_n + \mathbf{v}_{n+1})& =
   \mathbf{r}_n + h \mathbf{v}_n  +  \frac{h^2}{6}(\mathbf{f}_{n+1} + 2 \mathbf{f}_n) \nonumber\\
   &- \frac{h}{2}\left[ 2 \mathbf{v}_n + \frac{h}{2}(\mathbf{f}_n + \mathbf{f}_{n+1}) + {\cal O}(h^3) \right] + {\cal O}(h^4), \nonumber\\
   & =  \mathbf{r}_n + \frac{h^2}{12}(\mathbf{f}_n - \mathbf{f}_{n+1}) + {\cal O}(h^4),\text{ and, therefore,}\nonumber\\
   \mathbf{r}_{n+1} & = \frac{h}{2}(\mathbf{v}_n + \mathbf{v}_{n+1}) + \frac{h^2}{12}(\mathbf{f}_n - \mathbf{f}_{n+1})+ {\cal O}(h^4).\tag{20}
\end{align}

### Predictor/Corrector steps
Unfortunately, Eqs. (19) and (20) depend on future values of the velocity and the force per unit mass, $\mathbf{v}_{n+1}$  and $\mathbf{f}_{n+1}$, respectively! But happily there is a workaround that maintains the ${\cal O}(h^3)$ accuracy:

  1. predict $$\mathbf{r}_{n+1}^* = \mathbf{r}_{n+1} + {\cal O}(h^3)$$ using Eq.(15);
  2. compute the estimate $$\mathbf{f}_{n+1}^* = \mathbf{f}_{n+1} + {\cal O}(h^3)$$ using $\mathbf{r}_{n+1}^*$;
  3. correct the velocity predictions using $$\mathbf{v}_{n+1} = \mathbf{v}_n + h (\mathbf{f}_n + \mathbf{f}_{n+1}^*) / 2 + {\cal O}(h^3),$$ then
  4. correct the position predictions using $$\mathbf{r}_{n+1} = \mathbf{r}_n + h (\mathbf{v}_n + \mathbf{v}_{n+1}) / 2 + h^2 (\mathbf{f}_n - \mathbf{f}_{n+1}^*) / 12 + {\cal O}(h^3).$$

## Coordinate system
For this project, please choose the coordinate system so that $+z$ points upwards and the $x-y$ plane is horizontal. 

## References
 1. A. Rahman, *Correlations in the Motion of Atoms in Liquid Argon*, Phys. Rev. 136, A405, 1964.
 2. https://en.wikipedia.org/wiki/Lennard-Jones_potential
 3. https://chem.libretexts.org/Bookshelves/Physical_and_Theoretical_Chemistry_Textbook_Maps/Supplemental_Modules_(Physical_and_Theoretical_Chemistry)/Physical_Properties_of_Matter/Atomic_and_Molecular_Properties/Intermolecular_Forces/Specific_Interactions/Lennard-Jones_Potential

## Tips

  * Use __esc r__ to disable a cell
  * Use __esc y__ to reactivate it
  * Use __esc m__ to go to markdown mode
  * Shift + return to execute a cell

In [1]:
import numpy as np
from time import time, ctime

from types import SimpleNamespace as Bag

from comphyslab.newton import propagate_order3, propagate_order4, \
initialize_fcc, min_separation, TLennardJones, \
maxwell_distribution, radial_distribution, MDLoggerH5, KB

from comphyslab.vectors import magnitude, unit, dot, line_sphere_intersect

from comphyslab.utils import round_sig, round_sig_np

## Argon Parameters
We'll use the parameters for argon from Ref.[3] (but in SI units):
  * $\sigma = 3.4 \times 10^{-10} \text{ m}$;
  * $\epsilon =  1.657 \times 10^{-21} \text{ J}$;
  * $m = 6.69\times 10^{-26} \text{ kg}$, and
  * $\rho_0 = 1.642 \text{ kg/m}^3$ @ 293 K and one atmosphere. 

In [2]:
# A Rahman Parameters (Phys. Rev. 136, 1964)
liquid = Bag()
liquid.T = 94.4 # K
liquid.rho = 1.374e3  # kg/m^3

gas = Bag()
gas.T = 293.0 # K
gas.rho = 1.642       # kg/m^3
# --------------------------------------
def argon_parameters(rho=liquid.rho, T=liquid.T):   
    sigma   = 3.4e-10          # Distance at which potential is zero (m)
    epsilon = 1.657e-21        # Characteristic energy (J)
    mass    = 6.69e-26         # Mass of argon atom (kg)
    equisep = 2**(1/6)         # Equilibrium separation (sigma)
    v0 = float(np.sqrt(epsilon/mass)) # Characteristic speed (m/s)
    t0 = float(sigma / v0)            # Characteristic timescale (s)

    # Save in a bag
    bg = Bag()
    bg.sigma   = round_sig(sigma)
    bg.epsilon = round_sig(epsilon)
    bg.mass    = round_sig(mass)
    bg.v0      = round_sig(v0)
    bg.t0      = round_sig(t0)
    bg.T2K     = round_sig(mass * v0**2 / KB)

    print(f'''
    sigma:       {bg.sigma:10.2e} m
    epsilon:     {bg.epsilon:10.2e} J
    mass:        {bg.mass:10.2e} kg
    speed scale: {bg.v0:5.2f} m/s
    time scale:  {bg.t0:10.2e} s
    ''')
   
    # -------------------------------------------------------
    # Compute number density (units atoms/sigma**3)
    # -------------------------------------------------------
    print(f'requested number density: {rho:10.3e} kg/m^3')
    
    rho = rho / mass           # number of atoms/m^3
    print(f'requested number density: {rho:10.3e} atoms/m^3')
    
    rho = rho * sigma**3       # number of atoms/sigma**3
    print(f'requested number density: {rho:10.3e} atoms/sigma^3')
    
    print(f'requested temperature:    {T:6.3f} K\n')
    bg.rho = round_sig(rho)
    
    # -------------------------------------------------------
    # Create a lattice of points
    # -------------------------------------------------------
    ncells = 4
    r = initialize_fcc(ncells, full=True)
    print(f'number of lattice points generated: {len(r)}\n')
    N1 = len(r)
    V1 = N1/(2*rho)
    L  = V1**(1/3)
    r *= L
    
    # -------------------------------------------------------
    # Count how many atoms fit into the circumscribing sphere
    # -------------------------------------------------------
    Rc = L / np.sqrt(2)  
    rmag = magnitude(r)
    r = r[rmag < Rc]
    N = len(r)
    bg.N = N
    bg.r = r
    
    # -------------------------------------------------------
    # Compute sphere radius that yields the requested density
    # -------------------------------------------------------
    V = N / rho
    R = ((3.0/4/np.pi)*V)**(1/3)
    print(f'number atoms:    {N:5d} atoms in sphere of radius R = {R:6.3f} sigma')
    # sanity check
    rho_check = N / V
    print(f'number density:  {rho_check:10.3e} atoms/sigma^3')

    bg.R = round_sig(R)
    
    r_min_sep = min_separation(r)
    print(f'min(separation): {r_min_sep:6.3f} sigma (cf. {equisep:6.3f} sigma)\n')
    bg.r_min_sep = round_sig(r_min_sep)
    
    # -------------------------------------------------------
    # Generate initial velocities
    # -------------------------------------------------------
    vrms1 = np.sqrt(3*KB*T/mass)
    v = vrms1 * unit(np.random.uniform(-1.0, 1.0, 3*N).reshape(N, 3))
    
    # Remove center of mass motion and rescale velocities 
    # to arrive at specified temperature, defined as
    # T = (1/3)(m/KB) <v^2>
    if (len(v.shape) > 1) and (v.shape[0] > 1):
        v -= v.mean(axis=0)
    vrms2 = np.sqrt(np.mean((v**2).sum(axis=-1)))
    v *= vrms1 / vrms2
    bg.v = v
    
    # Check that we get the requested temperature
    vrms = float(np.sqrt(np.mean((v**2).sum(axis=-1))))
    T = (1/3)*(mass/KB)*vrms**2
    print(f'Vrms:  {vrms:5.1f} m/s,\tT: {T:8.2f} K')

    # Compute dimensionless velocities and dimensionless
    # temperature
    v /= v0
    vrms /= v0
    T_reduced = float(np.mean((v**2).sum(axis=-1)) / 3)
    print(f'Vrms:  {vrms:5.1f} v0,\tT: {T_reduced:8.2f}')
    bg.vrms = round_sig(vrms)

    return bg

argon = argon_parameters(rho=liquid.rho, T=liquid.T)


    sigma:         3.40e-10 m
    epsilon:       1.66e-21 J
    mass:          6.69e-26 kg
    speed scale: 157.38 m/s
    time scale:    2.16e-12 s
    
requested number density:  1.374e+03 kg/m^3
requested number density:  2.054e+28 atoms/m^3
requested number density:  8.072e-01 atoms/sigma^3
requested temperature:    94.400 K

number of lattice points generated: 365

number atoms:      297 atoms in sphere of radius R =  4.445 sigma
number density:   8.072e-01 atoms/sigma^3
min(separation):  1.077 sigma (cf.  1.122 sigma)

Vrms:  241.8 m/s,	T:    94.40 K
Vrms:    1.5 v0,	T:     0.79


##  Define the initial state of the system

In [3]:
def define_initial_state(argon):
    bg = Bag()
    bg.argon = argon
    # --------------------------------------------
    # Simulation Constants
    # --------------------------------------------
    bg.dt    = 1.e-3         # Time increment (units of t0) 

    # Container parameters
    bg.R     = argon.R       # Radius of container  (units of sigma)
    bg.origin= np.array([0.0, 0.0, 0.0])
    # --------------------------------------------
    # Particles
    # --------------------------------------------    
    # Initial positions
    bg.r  = argon.r
    bg.N  = len(bg.r)             # Number of particles
    bg.r0 = np.zeros_like(bg.r)
    
    # Initial velocities
    bg.v  = argon.v
    bg.v0 = np.zeros_like(argon.v)

    bg.mass  = np.ones(bg.N)      # Masses of particles (units of mass)
    bg.charge= np.ones(bg.N)      # Masses of atoms (units of mass)
    bg.rho   = argon.rho          # Atoms / sigma**3
    # --------------------------------------------
    # Force
    # --------------------------------------------
    bg.k = 1.0
    bg.law = TLennardJones
    
    return bg

## Compute bounce point

Given the current position $\vec{r}(t) = \vec{r}_0$ of a particle and its proposed position $\vec{r}(t+h) = \vec{r}_1$ determine whether it would cross the wall of the container during a given time step.  Let $\vec{r}_{10} = \vec{r}_1 - \vec{r}_0$ be the vector from $\vec{r}_0$ to $\vec{r}_1$ and let $\vec{u}$ be the unit vector that points in the same direction, that is,
\begin{align}
    \hat{u} & = \frac{\vec{r}_{10}}{r_{10}} .
\end{align}
The scalar $r_{10} = |\vec{r}_{10}|$ is the magnitude of $\vec{r}_{10}$, which is equal to the distance between $\vec{r}_1$ and $\vec{r}_0$.

The equation of a sphere centered as the orgin is $r = d$, where $d$ is the radius of the sphere and $r = |\vec{r}|$. 
![sphere/line intersection](sphere_line.png)

The equation of a line that passes through $\vec{r}_0$ is $\vec{r} = \vec{r}_0 + t \, \hat{u}$, where $t$ is a scalar that can be positive, zero, or negative. We expect the line to intersect the sphere at two points as shown in the figure above; therefore, we anticipate having to solve a quadratic equation. 
\begin{align}
    \vec{r} &= \vec{r}_0 + t \, \hat{u},\\
    \vec{r} \cdot \vec{r} & = 
  (\vec{r}_0 + t \, \hat{u} )\cdot(\vec{r}_0 + t \, \hat{u} )\\
   d^2 & = \vec{r}_0\cdot\vec{r}_0 + 2 t \, \hat{u}\cdot\vec{r}_0 + t^2 \text{ since } \hat{u}\cdot\hat{u} = 1.
\end{align}
As expected, we have arrived at a quadratic equation of the form $a \, t^2 + b \, t + c = 0$, where $a = 1$, $b = 2 \,\hat{u}\cdot\vec{r}_0$, and $c = \vec{r}_0\cdot\vec{r}_0 - d^2$. For the intersection point in the $-\hat{u}$ direction, that is, opposite the particle's velocity, $t$ will be negative and that point (the lower orange point to the right in the figure above) can be excluded. For the other intersection point $t$ is positive and is the distance from $\vec{r}_0$ to the intersection point denoted by $\vec{b}$ in the figure. The condition for deciding whether to bounce is now straightforward: *if $r_{10} \ge t$ then the particle would either touch or cross the wall and, therefore, should be made to bounce*.

In [4]:
def bounce_within_sphere(bg, eps=1e-12):
    """
    Specular elastic reflection from inner wall of sphere 
    radius bg.R centered at origin.
    """

    # Detect which predicted positions are outside
    r_mag  = magnitude(bg.r)                 # (n,)
    bounce = r_mag >= bg.R                   # (n,)
    if not np.any(bounce):
        bg.impulse = 0.0
        return

    # Subset of atoms to bounce
    r0 = bg.r0[bounce]                       # (k,3)
    v0 = bg.v0[bounce]                       # (k,3)
    r  = bg.r[bounce]                        # (k,3)
    v  = bg.v[bounce]                        # (k,3)

    # Direction of travel at start of step 
    # (for ray/sphere intersection)
    u0 = unit(v0)                            # (k,3)

    # Hit points on the sphere
    b, _, _ = line_sphere_intersect(r0, u0, bg.R)  # b: (k,3) points on |b|=R

    # Outward normal at hit points
    n = unit(b)                              # (k,3)

    # Normal speed at impact (outward positive). 
    # For actual "exiting" impacts should be > 0.
    vdotn = dot(v0, n)                       # (k,)

    # Reflect velocity direction about normal: v' = v - 2(v·n)n
    v_reflected = v0 - 2 * vdotn[:, None] * n    # (k,3)

    # Compute impulse (m=1) delivered to wall. This is just
    # the sum of the magnitudes of the change in momentum.
    # Protect against possible negative values due to float 
    # point rounding errors
    bg.impulse = 2.0 * float(np.sum(np.clip(vdotn, 0, None)))
    
    # Use predicted speed
    speed = magnitude(v)                     # (k,)

    # New velocities after bounce
    bg.v[bounce] = speed[:, None] * unit(v_reflected)

    # Move positions: keep "remaining distance" from hit point, 
    # but along reflected direction
    # Remaining distance traveled during step:
    s = magnitude(r - b)                     # (k,)

    # Nudge slightly inward along normal to avoid immediate 
    # re-trigger due to roundoff
    b_in = b - eps * n

    bg.r[bounce] = b_in + s[:, None] * unit(v_reflected)
     
def propagate(bg):
    # -----------------------------------------------------------
    # Update state of every particle.
    # -----------------------------------------------------------
    bg.r0[:] = bg.r # copy r into r0 before update
    bg.v0[:] = bg.v # copy v into v0 before update
    bg.r[:], bg.v[:] = propagate_order3(
        bg.k, bg.charge, bg.mass, bg.r0, bg.v0, bg.law, bg.dt)

def solve(bg):
    # Solve Newton's 2nd law for all particles in current time step, dt
    propagate(bg)
    bounce_within_sphere(bg)

## Compute

In [8]:
argon = argon_parameters(rho=1.05*liquid.rho, T=80.0)

bag = define_initial_state(argon)

logger = MDLoggerH5(
    'argon_solid.h5', bag.N, bag.dt, bag.R, bag.rho, save_every=10)

K = 15_000
impulse = 0.0
start_time = time()
try:
    for step in range(K):

        solve(bag)
        
        impulse += bag.impulse

        if step % logger.save_every == 0:
            print(f'\r{step:10d}\t{ctime()}', end='')
            logger.append(bag.r, bag.v, impulse)
            impulse = 0.0

        if step % 2000 == 0:
            logger.flush()
finally:
    logger.close()
    elapsed_time = time() - start_time
    time_per_iteration = 1e3 * elapsed_time / K
    print()
    print(f'time/step: {time_per_iteration:10.2f} ms')


    sigma:         3.40e-10 m
    epsilon:       1.66e-21 J
    mass:          6.69e-26 kg
    speed scale: 157.38 m/s
    time scale:    2.16e-12 s
    
requested number density:  1.443e+03 kg/m^3
requested number density:  2.157e+28 atoms/m^3
requested number density:  8.476e-01 atoms/sigma^3
requested temperature:    80.000 K

number of lattice points generated: 365

number atoms:      297 atoms in sphere of radius R =  4.373 sigma
number density:   8.476e-01 atoms/sigma^3
min(separation):  1.060 sigma (cf.  1.122 sigma)

Vrms:  222.6 m/s,	T:    80.00 K
Vrms:    1.4 v0,	T:     0.67
     14990	Wed Mar  4 23:28:39 2026
time/step:       9.84 ms
